# Regulatory Document Processing Pipeline with Textract

Universal document processing pipeline using AWS Textract and Bedrock for any file format.

## 1. Dependencies & Imports

Import necessary libraries for AWS integration (Boto3), NLP processing (Transformers, PyTorch), document parsing (BeautifulSoup), and data handling.

In [5]:
import boto3
import json
import time
import os
from typing import Dict, Any
from botocore.exceptions import ClientError
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModel, pipeline
import torch

/Users/rayanelgouri/Documents/Dev/ReguAI/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. RegulatoryDocumentProcessor Class

Main processing class that orchestrates:
- **Text Extraction**: Handles multiple formats (HTML, XML, PDF, images) using AWS Textract and BeautifulSoup
- **Classification**: Uses LegalBERT to classify regulations into categories (Environmental, Financial, Trade, Privacy, Labor, Tax)
- **Key Information Extraction**: Leverages Claude via Bedrock API to extract critical compliance details

In [9]:
class RegulatoryDocumentProcessor:
    def __init__(self):
        self.bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
        self.textract = boto3.client('textract', region_name='us-east-1')
        
        # Load LegalBERT for legal document classification
        print("📚 Loading LegalBERT model for legal analysis...")
        self.legal_tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.legal_model = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.legal_model.to(self.device)
        print("✅ LegalBERT model loaded successfully")
        
    def extract_text_from_file(self, file_path: str) -> str:
        """Extract text from any local file using Textract or BeautifulSoup"""
        print(f"📄 Extracting text from: {file_path.split('/')[-1]}")
        
        _, ext = os.path.splitext(file_path.lower())
        print(f"   File extension detected: {ext}")
        
        if ext in ['.html', '.xml']:
            # Use BeautifulSoup for HTML/XML FIRST (before Textract)
            print("   Using BeautifulSoup for HTML/XML parsing...")
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    soup = BeautifulSoup(file, 'html.parser')
                    text = soup.get_text(separator='\n')
                print(f"✅ Extracted {len(text):,} characters using BeautifulSoup")
                return text
            except Exception as e:
                print(f"❌ Error parsing HTML/XML: {e}")
                return ""
        
        elif ext in ['.pdf', '.jpg', '.jpeg', '.png', '.tiff', '.tif']:
            # Use Textract for images and PDFs
            print("   Using Textract for document extraction...")
            try:
                with open(file_path, 'rb') as file:
                    document_bytes = file.read()
                
                response = self.textract.detect_document_text(
                    Document={'Bytes': document_bytes}
                )
                
                text = ""
                for block in response['Blocks']:
                    if block['BlockType'] == 'LINE':
                        text += block['Text'] + "\n"
                
                print(f"✅ Extracted {len(text):,} characters using Textract")
                return text
            except ClientError as e:
                if e.response['Error']['Code'] == 'UnsupportedDocumentException':
                    print("❌ Unsupported document format. Textract supports JPEG, PNG, PDF, and TIFF files.")
                    return ""
                else:
                    raise
        
        else:
            print(f"❌ Unsupported file format: {ext}. Supported: PDF, images (JPEG/PNG/TIFF), HTML, XML.")
            return ""
    
    def classify_regulation_with_legalbert(self, text: str) -> Dict[str, Any]:
        """Classify regulation using LegalBERT for legal document understanding"""
        print("🔍 Classifying document using LegalBERT...")
        
        # Truncate text to avoid token limit (512 tokens max for BERT)
        truncated_text = text[:2000]
        
        try:
            # Tokenize input
            inputs = self.legal_tokenizer(
                truncated_text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True
            ).to(self.device)
            
            # Get embeddings from LegalBERT
            with torch.no_grad():
                outputs = self.legal_model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1)
            
            # Use embedding norm as confidence measure
            confidence = min(torch.norm(embeddings).item() / 10, 1.0)
            print(f"📋 LegalBERT confidence score: {confidence:.2%}")
            
            # Use Claude for contextual classification
            prompt = f"""Based on this regulatory document excerpt, classify it into ONE category:
            - Environmental
            - Financial
            - Trade
            - Privacy
            - Labor
            - Tax
            
            Document excerpt: {truncated_text}
            
            Return only the category name and a brief 1-2 sentence explanation."""
            
            response = self.bedrock.invoke_model(
                modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
                body=json.dumps({
                    'anthropic_version': 'bedrock-2023-05-31',
                    'max_tokens': 100,
                    'messages': [{'role': 'user', 'content': prompt}]
                })
            )
            
            result = json.loads(response['body'].read())
            classification = result['content'][0]['text'].strip()
            print(f"📋 Document classification: {classification}")
            
            return {
                'category': classification,
                'legalbert_confidence': confidence
            }
        
        except Exception as e:
            print(f"⚠️ Classification error: {e}. Falling back to basic extraction.")
            return {'category': 'Unknown', 'legalbert_confidence': 0.0}
    
    def extract_key_info(self, text: str, category: str) -> Dict[str, Any]:
        """Extract key regulatory information using Claude"""
        print(f"🔎 Extracting key information for {category} regulation...")
        prompt = f"""Extract key information from this {category} regulation. Return all information in English.
        
        Required fields:
        - title: Document title (translate to English if needed)
        - country: Country or region where this regulation applies (e.g., "United States", "European Union", "Japan", etc.)
        - effective_date: When it takes effect
        - affected_sectors: Which business sectors are impacted
        - key_requirements: Main compliance requirements (max 3)
        - penalties: Potential penalties for non-compliance
        
        Document: {text[:4000]}
        
        Return ONLY valid JSON (no markdown, no code blocks, just pure JSON). Ensure all text fields are in English."""
        
        response = self.bedrock.invoke_model(
            modelId='anthropic.claude-3-haiku-20240307-v1:0',
            body=json.dumps({
                'anthropic_version': 'bedrock-2023-05-31',
                'max_tokens': 500,
                'messages': [{'role': 'user', 'content': prompt}]
            })
        )
        
        result = json.loads(response['body'].read())
        content = result['content'][0]['text']
        print("✅ Key information extracted")
        
        try:
            return json.loads(content)
        except:
            return {'raw_content': content}
    
    def process_document(self, file_path: str) -> Dict[str, Any]:
        """Process any document format"""
        print(f"🚀 Starting document processing pipeline...")
        text = self.extract_text_from_file(file_path)
        
        if not text:
            print("❌ Failed to extract text from document")
            return None
        
        classification = self.classify_regulation_with_legalbert(text)
        key_info = self.extract_key_info(text, classification['category'])
        print("🎉 Processing completed successfully!")
        
        return {
            'document_id': file_path.split('/')[-1],
            'category': classification['category'],
            'legalbert_confidence': classification.get('legalbert_confidence', 0.0),
            'extracted_info': key_info,
            'processing_status': 'completed'
        }

## Process Any Document Format

## 3. Execute Full Processing Pipeline

Run the complete document processing workflow on a sample regulatory file. The pipeline will:
1. Locate the document file
2. Extract text based on file format
3. Classify the regulation type
4. Extract key compliance information
5. Return structured results

In [7]:
# Update the document_path to point to a valid local file and add a simple fallback search.
# The pipeline now supports HTML/XML alongside PDF and image formats.
# Adjust the candidate paths list if your files live elsewhere

candidate_paths = [
    os.path.join('../../data/raw/directives', '1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html'),
    # os.path.join('directive', '1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html'),

]

document_path = None
for path in candidate_paths:
    if os.path.exists(path):
        document_path = path
        break

if not document_path:
    raise FileNotFoundError(
        "Unable to locate the sample document. Update candidate_paths with a valid file location."
    )

print(f"Using document: {document_path}")

processor = RegulatoryDocumentProcessor()
try:
    result = processor.process_document(document_path)
except FileNotFoundError:
    print(f"Error: File not found at {document_path}. Please check the path and ensure the file exists.")
    result = None

if result:
    print("\nProcessing Results:")
    print(f"Document: {result['document_id']}")
    print(f"Category: {result['category']}")
    print(f"Status: {result['processing_status']}")
    print("\nExtracted Information:")
    print(json.dumps(result['extracted_info'], indent=2))

Using document: ../../data/raw/directives/1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html
📚 Loading LegalBERT model for legal analysis...
✅ LegalBERT model loaded successfully
🚀 Starting document processing pipeline...
📄 Extracting text from: 1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html
   File extension detected: .html
   Using BeautifulSoup for HTML/XML parsing...
✅ Extracted 111,996 characters using BeautifulSoup
🔍 Classifying document using LegalBERT...
✅ LegalBERT model loaded successfully
🚀 Starting document processing pipeline...
📄 Extracting text from: 1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html
   File extension detected: .html
   Using BeautifulSoup for HTML/XML parsing...
✅ Extracted 111,996 characters using BeautifulSoup
🔍 Classifying document using LegalBERT...
📋 LegalBERT confidence score: 90.80%
⚠️ Classification error: An error occurred (ValidationException) when calling the InvokeModel operation: Invocatio

In [10]:
# Process all files in the directives directory and export to JSON
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
env_path = Path.cwd()
if 'notebooks' in str(env_path):
    env_path = env_path.parent
env_file = env_path / '.env'
if env_file.exists():
    load_dotenv(env_file)
    print("✅ AWS Credentials loaded from .env")
else:
    print("⚠️ .env file not found. Using system environment variables.")

# Set output directory for regulatory files
output_dir = Path('../../data/generated/extracted_directives')
output_dir.mkdir(parents=True, exist_ok=True)
print(f"📁 Output directory: {output_dir}")

directives_dir = Path('../../data/raw/directives')

if directives_dir.exists():
    # Get all HTML/XML files in the directives directory
    all_files = list(directives_dir.glob('*.html')) + list(directives_dir.glob('*.xml'))
    
    if all_files:
        print(f"📋 Found {len(all_files)} regulatory files to process")
        print("=" * 80)
        
        results = []
        
        for idx, file_path in enumerate(all_files, 1):
            print(f"\n🔄 Processing {idx}/{len(all_files)}: {file_path.name}")
            print("-" * 80)
            
            try:
                processor = RegulatoryDocumentProcessor()
                result = processor.process_document(str(file_path))
                
                if result:
                    results.append(result)
                    
                    # 💾 Save individual JSON file
                    file_stem = file_path.stem
                    output_file = output_dir / f"{file_stem}_extracted.json"
                    with open(output_file, 'w', encoding='utf-8') as f:
                        json.dump(result, f, indent=2, ensure_ascii=False)
                    print(f"✅ Successfully processed and saved: {file_path.name}")
                    print(f"   📄 Saved to: {output_file}")
                else:
                    print(f"⚠️ Failed to process: {file_path.name}")
            
            except Exception as e:
                print(f"❌ Error processing {file_path.name}: {str(e)[:200]}")
                continue
        
        print("\n" + "=" * 80)
        print(f"📊 SUMMARY: Processed {len(results)}/{len(all_files)} files successfully")
        print("=" * 80)
        
        # 📦 Save consolidated results
        if results:
            # Save consolidated JSON with all results
            consolidated_file = output_dir / "all_directives_extracted.json"
            consolidated_dict = {result['document_id']: result for result in results}
            with open(consolidated_file, 'w', encoding='utf-8') as f:
                json.dump(consolidated_dict, f, indent=2, ensure_ascii=False)
            print(f"\n📦 Consolidated results saved to: {consolidated_file}")
            
            # Save CSV summary
            import pandas as pd
            rows = []
            for result in results:
                row = {
                    'document_id': result['document_id'],
                    'category': result['category'],
                    'confidence': result['legalbert_confidence'],
                    'status': result['processing_status'],
                    'title': result['extracted_info'].get('title', ''),
                    'effective_date': result['extracted_info'].get('effective_date', ''),
                    'affected_sectors': '; '.join(result['extracted_info'].get('affected_sectors', [])) if isinstance(result['extracted_info'].get('affected_sectors'), list) else result['extracted_info'].get('affected_sectors', ''),
                }
                rows.append(row)
            
            df = pd.DataFrame(rows)
            csv_file = output_dir / "directives_summary.csv"
            df.to_csv(csv_file, index=False)
            print(f"📊 Summary CSV saved to: {csv_file}")
        
        # Display summary of all results
        print("\n📋 DETAILED RESULTS:")
        print("-" * 80)
        for result in results:
            print(f"\n📄 {result['document_id']}")
            print(f"   Category: {result['category']}")
            print(f"   Confidence: {result['legalbert_confidence']:.2%}")
            print(f"   Status: {result['processing_status']}")
    else:
        print("⚠️ No HTML or XML files found in the directives directory")
else:
    print(f"❌ Directives directory not found: {directives_dir}")

⚠️ .env file not found. Using system environment variables.
📁 Output directory: ../../data/generated/extracted_directives
📋 Found 5 regulatory files to process

🔄 Processing 1/5: 4.REGULATION (EU) 20241689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL.html
--------------------------------------------------------------------------------
📚 Loading LegalBERT model for legal analysis...
✅ LegalBERT model loaded successfully
🚀 Starting document processing pipeline...
📄 Extracting text from: 4.REGULATION (EU) 20241689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL.html
   File extension detected: .html
   Using BeautifulSoup for HTML/XML parsing...
✅ Extracted 621,123 characters using BeautifulSoup
🔍 Classifying document using LegalBERT...
📋 LegalBERT confidence score: 100.00%
⚠️ Classification error: An error occurred (ValidationException) when calling the InvokeModel operation: Invocation of model ID anthropic.claude-sonnet-4-5-20250929-v1:0 with on-demand throughput isn’t supported. Retry 

/var/folders/vm/0c_1gypx2vqdqfgv08ckhl300000gn/T/ipykernel_46279/2593516941.py:26: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(file, 'html.parser')


✅ Extracted 758,277 characters using BeautifulSoup
🔍 Classifying document using LegalBERT...
📋 LegalBERT confidence score: 89.05%
⚠️ Classification error: An error occurred (ValidationException) when calling the InvokeModel operation: Invocation of model ID anthropic.claude-sonnet-4-5-20250929-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.. Falling back to basic extraction.
🔎 Extracting key information for Unknown regulation...
✅ Key information extracted
🎉 Processing completed successfully!
✅ Successfully processed and saved: 3.H.R.5376 - Inflation Reduction Act of 2022.xml
   📄 Saved to: ../../data/generated/extracted_directives/3.H.R.5376 - Inflation Reduction Act of 2022_extracted.json

📊 SUMMARY: Processed 5/5 files successfully

📦 Consolidated results saved to: ../../data/generated/extracted_directives/all_directives_extracted.json
📊 Summary CSV saved to: ../../data/generated/extracted_direct